# TN4 — Test cuối trên GHIJ, DS-TCN 64 kênh tầm nhìn 121

Train đủ **ABCDEFKL**, chấm **một lần** trên **GHIJ** — 537 buổi ghi của 4 người
chưa từng tham gia bất kỳ bước chọn cấu hình nào. Đây là **số công bố**, theo
`docs/PROTOCOL.md` mục 6.

## Cấu hình

| | |
|---|---|
| model | `ds_tcn --channels 64 --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **38.105** |
| tầm nhìn | **121** — phủ 60% cửa sổ vào, và 121% một pha thở |
| dev, MSE thuần | 0,757855 *(TN2, 4 fold, 1 seed)* |

## Vì sao tầm nhìn 121

TN2 chia bốn mức tầm nhìn thành hai nhóm tách rõ:

| tầm nhìn | c64 | c192 | |
|---:|---:|---:|---|
| 61 | 0,760878 | 0,762714 | nhóm tốt |
| **121** | **0,757855** | **0,764428** | **nhóm tốt** |
| 181 | 0,743657 | 0,732562 | nhóm tệ |
| 241 | 0,736970 | 0,736623 | nhóm tệ |

Hai nhóm cách nhau **0,0142**, lớn hơn dao động giữa các seed (0,0007–0,0108)
nên đọc được. Trong nhóm tốt chỉ chênh **0,0030**, nhỏ hơn dao động seed nên
**không xếp hạng được** — ở c64 thì 61 nhỉnh hơn, ở c192 thì 121 nhỉnh hơn, cả
hai đều dưới nhiễu.

Chọn 121 theo một tiêu chí thiết kế nêu được trước khi nhìn kết quả: nhịp thở
0,25 Hz lấy mẫu 50 Hz nên **một chu kỳ thở đúng 200 mẫu, một pha 100 mẫu**. Tầm
nhìn 61 chỉ phủ 61% một pha, chưa nhìn trọn một lần hít vào; **121 là mức nhỏ
nhất trong các mức đã thử phủ trọn một pha**. Đúng tinh thần Bai et al. mục A.1:
chọn `k` và `d` sao cho tầm nhìn phủ đủ ngữ cảnh mà bài toán cần.

Không viết "tầm nhìn 121 tốt hơn 61" — số liệu ở c64 nói ngược lại.

## Hai nhóm chạy, mỗi nhóm ba seed

**Mục 3 — loss lai alpha 0,0**, tức Pearson thuần. Mức này là **đỉnh TN3 của
chính cấu hình này**: quét mười mức cho `Ours-64/121` thì alpha 0,0 đạt
**0,780306**, cao nhất, hơn MSE thuần 0,0225.

Cùng quy tắc đã dùng cho hai lần chạy TN4 kia — mỗi cấu hình lấy đỉnh TN3 của
chính nó: `Ours-64/61` lấy 0,6 và `Ours-192/121` lấy 0,2.

**Vì sao không lấy 0,6 như `Ours-64/61`.** Ở cấu hình này alpha 0,6 cho
**0,752386 — thấp nhất trong mười mức, và là mức duy nhất trong cả đồ án thua
MSE thuần**. Hai cấu hình chỉ khác nhau tầm nhìn mà đỉnh nhảy từ 0,6 sang 0,0.
Chạy test ở mức đáy sẽ ra một con số thấp một cách vô nghĩa.

Đánh đổi: lần chạy này khác `TN4 Ours-64/61` **hai biến** — tầm nhìn và alpha —
nên không so trực tiếp hai con số GHIJ với nhau được. Phải chọn: hoặc giữ phép
so một biến nhưng chạy ở mức đáy, hoặc dùng đỉnh của chính cấu hình và mất phép
so. Chọn cái sau, vì số công bố quan trọng hơn.

**Mục 4 — MSE thuần.** Cùng kiến trúc, chỉ đổi hàm loss. Đây là thứ đang thiếu ở
mọi cấu hình khác: không có nó thì **không nói được hàm loss lai có giúp trên
tập test hay không**, chỉ nói được là nó giúp trên tập phát triển.

Mỗi lần chạy khoảng **20 phút** *(train 20 epoch ~11,5 phút, chấm 537 buổi ghi
~9 phút)*. Mỗi nhóm ba seed ≈ **1 giờ**, cả hai nhóm ≈ **2 giờ**.

## Một quy tắc phải giữ

`alpha 0,0` đã chốt **trước khi chạy**, lấy từ TN3 trên dev. Nếu về sau
`TN3_HybridLoss_DS_TCN_c64_rf121` chạy xong và cho đỉnh ở mức khác, mà bạn chạy
tiếp mức đó rồi **lấy con số GHIJ cao hơn trong hai cái** — thì GHIJ đã thành
tập chọn cấu hình và toàn bộ giao thức hỏng. Chạy thêm thì phải **báo cáo cả
hai**, kèm lý do vì sao có hai lần chạy.

`run_final_test.py` tự nén và chép sang Drive sau mỗi lần chạy nên không cần ô
lưu riêng. Dừng giữa chừng cũng được: mở lại, chạy ô khôi phục ở mục 1 rồi bấm
tiếp seed còn thiếu.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Test cuối đọc `windows/final_train/`, cắt gộp cả 8 người theo đúng thứ tự MobiVital.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục các seed đã chạy.

**Chạy ô này mỗi khi mở lại notebook.** Mẫu tên tệp bắt riêng `c64_k5_`, không lẫn với tệp của notebook `c64` tầm nhìn 61.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn4_*c64_k5_*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **38.105**.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Loss lai alpha 0,0 — ba seed

**seed 0**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 0

**seed 1**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 1

**seed 2**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 2

## 4. MSE thuần — ba seed

Cùng kiến trúc, chỉ đổi hàm loss. Hiệu giữa mục 3 và mục 4 là **đóng góp của hàm loss lai đo trên tập test**.

**seed 0**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse --seed 0

**seed 1**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse --seed 1

**seed 2**

In [ ]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse --seed 2

## 5. Kết quả

`compare_cv.py --final` gộp các seed của cùng một cấu hình thành `mean ± std`.
Nó nhận ra các lần chạy cùng cấu hình bằng cách bỏ hậu tố `_seed<N>` khỏi
`run_id`, nên hai nhóm ở mục 3 và mục 4 tự tách thành hai dòng.

Cột điểm là **macro** — trung bình theo người. Muốn xem cả **micro** thì mở
`notebooks/BANG_DIEM_MICRO_MACRO.ipynb`, nó đọc thẳng `score_micro` từ Drive.

**Mốc đối chiếu trên GHIJ**, cùng pipeline và cùng ba seed:

| | tham số | macro |
|---|---:|---:|
| LSTM-352 *(kiến trúc MobiVital)* | 1.502.713 | 0,810302 ± 0,015403 |
| Ours-64/61 alpha 0,6 | 37.081 | 0,803590 ± 0,015350 |
| Ours-192/121 alpha 0,2 | 310.873 | 0,800721 ± 0,010705 |
| DS-TCN-nền, MSE | 56.281 | 0,795782 ± 0,015413 |

In [ ]:
!python scripts/compare_cv.py --experiment tn4 --final

## 6. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()